## Sanskrit Machine Learning!

 Using NLP to learn embeddings from Vedic texts and explore conceptual similarity between verses, deities, and philosophical ideas (Dharma, Rta, Atman, Brahman).


### Loading the Dataset:
#### This is for the Kaggle Dataset!


Imports needed for Kaggle

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

In [2]:
# This is the file INSIDE the Kaggle dataset
file_path = "complete_rigveda_all_mandalas.json"

# Load the dataset as a pandas DataFrame
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "varunrajuvangar/rigved-all-sukta-verses-and-meaning-dataset",
    file_path,
)

print("\nColumns:")
print(df.columns)


Columns:
Index(['Mandala 1', 'Mandala 2', 'Mandala 3', 'Mandala 4', 'Mandala 5',
       'Mandala 6', 'Mandala 7', 'Mandala 8', 'Mandala 9', 'Mandala 10'],
      dtype='object')


In [3]:
# Display the first Sukta data to verify content and structure for cleaning
import pprint
first_sukta_data = df.iloc[0,0]
print("\nFirst Sukta Data:")
pprint.pprint(first_sukta_data)



First Sukta Data:
[{'padapatha': {'devanagari': {'text': 'अग्निम् । ईळे । पुरःऽहितम् । यज्ञस्य । '
                                       'देवम् । ऋत्विजम् ।होतारम् । '
                                       'रत्नऽधातमम् ॥',
                               'type': 'Padapatha Devanagari Nonaccented',
                               'words': ['अग्निम्',
                                         'ईळे',
                                         'पुरःऽहितम्',
                                         'यज्ञस्य',
                                         'देवम्',
                                         'ऋत्विजम्',
                                         'होतारम्',
                                         'रत्नऽधातमम्']},
                'transliteration': {'text': 'agním ǀ īḷe ǀ puráḥ-hitam ǀ '
                                            'yajñásya ǀ devám ǀ ṛtvíjam '
                                            'ǀhótāram ǀ ratna-dhā́tamam ǁ',
                                    'type': 'Padap

In [4]:
every_verse = []

for mandala in df.columns:
    for sukta in df.index:
        curr_cell = df.at[sukta, mandala]

        if not isinstance(curr_cell, list):
            continue

        for verse in curr_cell:
            try:
                sanskrit_verse = verse['samhita']['devanagari']['text']
                display_sanskrit = verse['sanskrit_wisdomlib']
                eng_translation = verse['translation']
                verse_num = verse['rik_number']

                every_verse.append({
                    'mandala': mandala,
                    'sukta': sukta,
                    'verse_num': verse_num,
                    'sanskrit_verse': sanskrit_verse,
                    'display_sanskrit': display_sanskrit,
                    'english_translation': eng_translation
                })

            except KeyError as e:
                print(f"KeyError for mandala {mandala}, sukta {sukta}, verse {verse.get('rik_number', '?')}: {e}")

cleaned_df = pd.DataFrame(every_verse)
cleaned_df = cleaned_df.dropna(subset=['sanskrit_verse', 'english_translation'])
cleaned_df = cleaned_df[cleaned_df['english_translation'].str.strip().str.len() > 5]
cleaned_df = cleaned_df.reset_index(drop=True)

print("\nCleaned DataFrame head:")
print(cleaned_df.head())


Cleaned DataFrame head:
     mandala    sukta  verse_num  \
0  Mandala 1  Sukta 1          1   
1  Mandala 1  Sukta 1          2   
2  Mandala 1  Sukta 1          3   
3  Mandala 1  Sukta 1          4   
4  Mandala 1  Sukta 1          5   

                                      sanskrit_verse  \
0  अग्निमीळे पुरोहितं यज्ञस्य देवमृत्विजं ।होतारं...   
1  अग्निः पूर्वेभिर्ऋषिभिरीड्यो नूतनैरुत ।स देवाँ...   
2  अग्निना रयिमश्नवत्पोषमेव दिवेदिवे ।यशसं वीरवत्...   
3  अग्ने यं यज्ञमध्वरं विश्वतः परिभूरसि ।स इद्देव...   
4  अग्निर्होता कविक्रतुः सत्यश्चित्रश्रवस्तमः ।दे...   

                                    display_sanskrit  \
0  अ॒ग्निमी॑ळे पु॒रोहि॑तं य॒ज्ञस्य॑ दे॒वमृ॒त्विज॑...   
1  अ॒ग्निः पूर्वे॑भि॒ॠषि॑भि॒रीड्यो॒ नूत॑नैरु॒त । ...   
2  अ॒ग्निना॑ र॒यिम॑श्नव॒त्पोष॑मे॒व दि॒वेदि॑वे । य...   
3  अग्ने॒ यं य॒ज्ञम॑ध्व॒रं वि॒श्वत॑: परि॒भूरसि॑ ।...   
4  अ॒ग्निर्होता॑ क॒विक्र॑तुः स॒त्यश्चि॒त्रश्र॑वस्...   

                                 english_translation  
0  “I glorifyAgni, the high p

In [5]:
print("Starting OCR text cleanup...")

# 1. Edge-Case Character & Specific OCR Fixes
ocr_fixes = {
    r'\bplural ce\b': 'place',
    r'\bplural ced\b': 'placed',
    r'\bplural asure\b': 'pleasure',
    r'\bplural asant\b': 'pleasant',
    r'\bplural ased\b': 'pleased',
    r'\bplural ant\b': 'plant',
    r'\bfeminine les\b': 'females',
    r'\bshharp\b': 'sharp',
    r'\btun turns\b': 'sun turns',
    r'\bnad by\b': 'and by',
    r'\bthelunar\b': 'the lunar',
    r'\bthethreefold\b': 'the threefold',
    r'\btheadorable\b': 'the adorable',
    r'\bwom who\b': 'women who',
    r'\bhisfoes\b': 'his foes',
    r'\baredecorated\b': 'are decorated',
    r'\bexhilaratingcelebrator\b': 'exhilarating celebrator',
    r'\bpossexceeding\b': 'possess exceeding',
    r'\bfoot-markof\b': 'foot-mark of',
    r'\bwhosehandsin\b': 'whose hands in',
    r'\bhismight\b': 'his might',
    r'\btheeggs\b': 'the eggs',
    r'\btheocean\b': 'the ocean',
    r'\btheprosperous\b': 'the prosperous',
    r'\btheuniversal\b': 'the universal',
    r'\bthemighty\b': 'the mighty',
    r'\bwithmilk\b': 'with milk',
    r'\bplural ntifully\b': 'plentifully',
    r'\byourvigour\b': 'your vigour',
    r'\bwiththe\b': 'with the',
    r'\btheirhearts\b': 'their hearts',
    r'\binthis\b': 'in this',
    r'\bbythe\b': 'by the',
    r'\bofwater\b': 'of water',
    r'\bofearth\b': 'of earth',
    r'\bplural nts\b': 'plants',
    r'\bmaythe\b': 'may the',
    r'\byourhandfor\b': 'your hand for',
    r'\bffrom\b': 'from',
    
}
cleaned_df['english_translation'] = cleaned_df['english_translation'].replace(ocr_fixes, regex=True)

# 2. Lowercase to Uppercase Splits (Including Sanskrit Diacritics)
cleaned_df['english_translation'] = cleaned_df['english_translation'].str.replace(
    r'([a-zāīūṛśṣṭḍṇṃḥ])([A-ZĀĪŪṚŚṢṬḌṆႱ])', 
    r'\1 \2', 
    regex=True
)

# 3. Punctuation and Parentheses Isolation
cleaned_df['english_translation'] = cleaned_df['english_translation'].str.replace(
    r'([\,\.\!\?\;\:\)])([a-zA-ZĀ-Ƶā-ž])', 
    r'\1 \2', 
    regex=True
)
cleaned_df['english_translation'] = cleaned_df['english_translation'].str.replace(
    r'([a-zA-ZĀ-Ƶā-ž])(\()', 
    r'\1 \2', 
    regex=True
)

# 4. Hardened Vedic Proper Noun Boundary Enforcement
deities = (
    r'(Agni|Indra|Soma|Vṛtra|Vritra|Manu|Varuna|Varuṇa|Mitra|Rudra|Vishnu|'
    r'Maruts|Ashvins|Arbuda|Āditya|Vasus|Mātariśvan|Tvaṣṭā|Ṛbhu|Nahuṣa|Bala|Trita|Aṅgirasa'
    r'Indrāṇī|Brāhmaṇas|Vāyu)'
)

cleaned_df['english_translation'] = cleaned_df['english_translation'].str.replace(
    rf'{deities}([a-zāīūṛśṣṭḍṇṃḥ]+)', 
    r'\1 \2', 
    regex=True
)
cleaned_df['english_translation'] = cleaned_df['english_translation'].str.replace(
    rf'([a-zāīūṛśṣṭḍṇṃḥ]+){deities}', 
    r'\1 \2', 
    regex=True
)


print("Cleanup pipeline executed successfully!")
print("Sanity Check of first few translations:")
print(cleaned_df['english_translation'].head(10))

Starting OCR text cleanup...
Cleanup pipeline executed successfully!
Sanity Check of first few translations:
0    “I glorify Agni, the high priest of the sacrif...
1    “May that Agni who is to be celebrated by both...
2    “Through Agni the worshipper obtains that affl...
3    “Agni, the unobstructed sacrifice of which you...
4    “May Agni, the presenter of oblations, the att...
5    “Whatever good you may, Agni, bestow upon the ...
6    “We approach you, Agni, with reverential homag...
7    “You, the radiant, the protector of sacrifice,...
8    “Agni, be to us easy of access, as is a father...
9    “Vāyu, pleasant to behold, approach; these lib...
Name: english_translation, dtype: object


## Embeddings for English Translations

In [6]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded successfully!")

Model loaded successfully!


In [7]:
print("Generating embeddings...")
embeddings = model.encode(
    cleaned_df['english_translation'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"Embeddings shape: {embeddings.shape}")
cleaned_df['embedding'] = list(embeddings)

Generating embeddings...


Batches:   0%|          | 0/326 [00:00<?, ?it/s]

Embeddings shape: (10402, 384)


In [8]:
# 1. Look at a single verse embedding
print("Single verse embedding (first 10 dimensions):")
print(embeddings[0][:10])
print(f"\nFull embedding shape for one verse: {embeddings[0].shape}")

# 2. See embeddings for first 3 verses
print("\nFirst 3 verse embeddings:")
print(embeddings[:3])

# 3. Compare two verse embeddings side-by-side
print("\nComparing verse 0 and verse 1:")
print(f"Verse 0 embedding: {embeddings[0][:5]}...")
print(f"Verse 1 embedding: {embeddings[1][:5]}...")

Single verse embedding (first 10 dimensions):
[ 0.02038838  0.75208515 -0.140578    0.24916014 -0.04747223 -0.27098244
  0.8612643  -0.14414275  0.19106732 -0.28578034]

Full embedding shape for one verse: (384,)

First 3 verse embeddings:
[[ 0.02038838  0.75208515 -0.140578   ... -0.05781809  0.10634226
   0.19520652]
 [ 0.18221973  0.6163792  -0.08789028 ...  0.12031072  0.21780422
   0.22670391]
 [ 0.1402334   0.52247524 -0.26468393 ... -0.12264356  0.13136435
   0.22433376]]

Comparing verse 0 and verse 1:
Verse 0 embedding: [ 0.02038838  0.75208515 -0.140578    0.24916014 -0.04747223]...
Verse 1 embedding: [ 0.18221973  0.6163792  -0.08789028  0.22212848 -0.12735493]...


In [9]:
# Preparation (Do this once):
# You have your DataFrame cleaned_df.
# You have a column embeddings which contains the vectors for every verse.
# Crucial Step: Extract all those individual vectors from the DataFrame and stack them into one big block (a matrix). This speeds up the search process.
verse_matrix = np.vstack(cleaned_df['embedding'].values)
print(f"\nVerse matrix shape: {verse_matrix.shape}")
print(verse_matrix[:2, :5])  

# The Search Function (Run this every time you search):
# Input: Take a text string (the "Query") from the user.
# Vectorize: Feed that text string into your SentenceTransformer model. This spits out a single vector (list of numbers).
# Math: Compare that Single Query Vector against the Big Matrix of Verse Vectors.
# Score: The result will be a list of 10,000 scores (between -1 and 1).
# Assign: Paste these scores back into your DataFrame as a new temporary column called "similarity_score".
# Sort: Sort the DataFrame so the rows with the highest "similarity_score" are at the top.
# Slice: Cut off the top 5 or 10 rows.
# Output: Print the English translation and Sanskrit text for those top rows.


Verse matrix shape: (10402, 384)
[[ 0.02038838  0.75208515 -0.140578    0.24916014 -0.04747223]
 [ 0.18221973  0.6163792  -0.08789028  0.22212848 -0.12735493]]


In [10]:
def search_verses(query, top_k):
    # Vectorize the query
    query_vector = model.encode([query])

    # Compute Similarity Scores
    similarity_scores = cosine_similarity(query_vector, verse_matrix).flatten()

    # Assign Scores to DataFrame
    cleaned_df['similarity_score'] = similarity_scores

    # Sort and get top K
    top_verses = cleaned_df.sort_values(by='similarity_score', ascending=False).head(top_k)

    # Output Results
    return top_verses[['mandala', 'sukta', 'verse_num', 'sanskrit_verse', 'english_translation', 'similarity_score']]

In [11]:
pd.set_option('display.max_colwidth', None)
search_verses("गणपति", top_k=10)

,mandala,sukta,verse_num,sanskrit_verse,english_translation,similarity_score
2197,Mandala 2,Sukta 19,8,एवा ते गृत्समदाः शूर मन्मावस्यवो न वयुनानि तक्षुः ।ब्रह्मण्यंत इंद्र ते नवीय इषमूर्जं सुक्षितिं सुम्नमश्युः ॥,"“Thus, hero, have the Gṛtsamadas”",0.575784
9739,Mandala 10,Sukta 95,17,अंतरिक्षप्रां रजसो विमानीमुप शिक्षाम्युर्वशीं वसिष्ठः ।उप त्वा रातिः सुकृतस्य तिष्ठान्नि वर्तस्व हृदयं तप्यते मे ॥,"“(Purūravā). I, Vasiṣṭha, bring under subjection Ūrvaśīwho fills the firmament (with lustre) andmeasures out the rain. May (Purūravā), the bestower of the auspicious rite, abide near you; come back-- myheart is burning.”",0.561787
2004,Mandala 2,Sukta 1,2,तवाग्ने होत्रं तव पोत्रमृत्वियं तव नेष्ट्रं त्वमग्निदृतायतः ।तव प्रशास्त्रं त्वमध्वरीयसि ब्रह्मा चासि गृहपतिश्च नो दमे ॥,"“Yours Agni, is the office of the Hotā, of the Potā, of the Ṛtvij, of the Neṣṭā; you are the Agnīdhraof the devout; yours is the functionof the Praśāstā; you are the Adhvaryu (adhvaryu radhvarayur adhvaram kāmayata iti vā (Nirukta1.8) and the Brahmā; and the householder in our dwelling.”",0.560247
1827,Mandala 1,Sukta 170,5,त्वमीशिषे वसुपते वसूनां त्वं मित्राणां मित्रपते धेष्ठः ।इंद्र त्वं मरुद्भिः सं वदस्वाध प्राशान ऋतुथा हवींषि ॥,"“(Agastya); You, Vasupati, are the lord of riches; you, Mitra pati, are the firm stay (of us), your friends; declare, Indra, along with the Maruts, (your approval of our acts), and partake of the oblation offered in due season.”",0.557471
1970,Mandala 1,Sukta 188,11,पुरोगा अग्निर्देवानां गायत्रेण समज्यते ।स्वाहाकृतीषु रोचते ॥,"“Agni, the preceder of the gods”",0.556448
7296,Mandala 8,Sukta 75,5,तं नेमिमृभवो यथा नमस्व सहूतिभिः ।नेदीयो यज्ञमंगिरः ॥,"“OAṅgirasa, with the deities associated in the invocation draw this offering near you as the Ṛbhu s (bend) the circumference of a wheel.”",0.549981
9673,Mandala 10,Sukta 91,10,तवाग्ने होत्रं तव पोत्रमृत्वियं तव नेष्ट्रं त्वमग्निदृतायतः ।तव प्रशास्त्रं त्वमध्वरीयसि ब्रह्मा चासि गृहपतिश्च नो दमे ॥,"“Yours, Agni, is the function of the Hotā, yours the duly-performed function of the Potā, yours thefunction of the Neṣṭā, you are the Agni of the sacrificer, yours is the office of the Praśāstā, you act as Adhvaryu, and you are the Brahmāand the lord of the mansion in our abode.”",0.547657
1102,Mandala 1,Sukta 100,16,रोहिच्छ्यावा सुमदंशुर्ललामीर्द्युक्षा राय ऋज्राश्वस्य ।वृषण्वंतं बिभ्रती धूर्षु रथं मंद्रा चिकेत नाहुषीषु विक्षु ॥,"“The red and black coursers, long-limbed, well-caparisoned, and celestial, an dharnessed, well-pleased, to the yoke of the chariot in which the showerer of benefits is conveyed, for the enrichment of Ṛjrāśva, an dis recognized amongst human hosts.”",0.547028
8150,Mandala 9,Sukta 64,22,इंद्रायेंदो मरुत्वते पवस्व मधुमत्तमः ।ऋतस्य योनिमासदं ॥,"“Flow, Indu, for Indra associated with the Maruts, you who are most sweet-flavoured, and take your seat on the place of the sacrifice.”",0.546295
7895,Mandala 9,Sukta 33,3,सुता इंद्राय वायवे वरुणाय मरुद्भ्यः ।सोमा अर्षंति विष्णवे ॥,"“The libations effused proceed to Indra, to Vāyu, to Varuṇa, to the Maruts, to Viṣṇu”",0.546116


In [12]:
print("Starting t-SNE dimensionality reduction (crushing 384 dimensions to 2)...")

# 1. Grab the fresh embeddings we just generated
embeddings_matrix = np.vstack(cleaned_df['embedding'].values)

# 2. Run the math to create the cluster coordinates
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
coords_2d = tsne.fit_transform(embeddings_matrix)

# 3. Attach the new coordinates directly to your dataframe
cleaned_df['x_coord'] = coords_2d[:, 0]
cleaned_df['y_coord'] = coords_2d[:, 1]

print("t-SNE Calculation Finished!")
print(cleaned_df[['english_translation', 'x_coord', 'y_coord']].head())

Starting t-SNE dimensionality reduction (crushing 384 dimensions to 2)...
t-SNE Calculation Finished!
                                                                                                                                              english_translation  \
0  “I glorify Agni, the high priest of the sacrifice, the divine, the ministrant, who presents the oblation (to the gods), and is the possessor of great wealth.”   
1                                                               “May that Agni who is to be celebrated by both ancient and modern sages conduct the gods hither.”   
2                         “Through Agni the worshipper obtains that affluence which increases day by day, which is the source of fame and multiplier of mankind.”   
3                                                    “Agni, the unobstructed sacrifice of which you are on every side the protector, assuredly reaches the gods.”   
4                             “May Agni, the presenter of oblations, the 

In [13]:

# cleaned_df.to_csv("rigveda_clean.csv", index=False)
cleaned_df.to_parquet("rigveda_clean.parquet")